# TP1 — EDA & Preprocessing initial
## Fil rouge : Churn Predictor (Telco Customer Churn)

**Durée estimée : 1h15**

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Charger et inspecter un dataset réel (types, valeurs manquantes, cardinalité).
- Détecter et corriger un piège classique de typage (`TotalCharges` stocké en string).
- Mener une EDA ciblée : distribution de la cible, variables numériques et catégorielles vs target.
- Construire un **split stratifié train / val / test** avant tout traitement (et comprendre pourquoi).

### 📦 Pré-requis
- Python 3.10+
- `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`
- Connexion internet (le dataset est chargé depuis GitHub)

### 📌 Contexte métier
Vous travaillez pour un opérateur télécom. Chaque mois, ~26 % des clients résilient. La direction veut un modèle qui identifie les clients à risque pour cibler une campagne de rétention. Avant de modéliser, on doit **comprendre la donnée**.


## 0. Setup

On importe les librairies et on fixe une seed pour la reproductibilité.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 1. Chargement du dataset

Le dataset Telco Customer Churn est disponible publiquement. On le charge directement depuis GitHub.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
)

# Load the CSV from DATA_URL into a DataFrame named `df`
df = pd.read_csv(DATA_URL)

# Print the shape and display the first 5 rows
print(f"Shape: {df.shape}")
df.head()


## 2. Premier regard

Avant toute analyse, on inspecte systématiquement :
- Les **types** de chaque colonne (`dtypes`)
- Les **valeurs manquantes** (`isna().sum()`)
- La **cardinalité** de chaque colonne (`nunique()`) — utile pour distinguer numérique / catégoriel / identifiant

⚠️ **Point de vigilance** : ne JAMAIS faire confiance aux types par défaut de pandas. Toujours vérifier.

In [ ]:
# Display column info (dtypes, non-null counts) using df.info()
df.info()

# Display the count of missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

# Display the number of unique values per column, sorted ascending
# This helps you spot identifiers (high cardinality) and binary cols (2 unique values)
print("\nUnique values per column (sorted ascending):")
print(df.nunique().sort_values())


## 3. Le piège de `TotalCharges`

Regardez bien le résultat de `df.info()` à la cellule précédente. Vous devriez constater que **`TotalCharges` est de type `object`** alors qu'il devrait être numérique (c'est un montant en dollars).

Pourquoi ? Parce que certaines lignes contiennent une chaîne vide (`" "`) au lieu d'un nombre. C'est un bug classique de l'export.

🎯 **Tâche** :
1. Convertir `TotalCharges` en numérique avec `errors="coerce"` (ce qui transforme `" "` en `NaN`).
2. Compter combien de NaN apparaissent.
3. Inspecter ces lignes : qu'ont-elles de particulier ?
4. Décider d'une stratégie de remplissage.

In [ ]:
# Convert TotalCharges to numeric (use errors="coerce")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# How many NaN do we have now?
print(f"NaN count: {df['TotalCharges'].isna().sum()}")

# Display the rows where TotalCharges is NaN — focus on tenure and MonthlyCharges
# What do these clients have in common? Write your observation as a markdown comment.
df[df["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges"]]


💡 **Observation** : tous les clients avec `TotalCharges` manquant ont `tenure == 0`. Ce sont des nouveaux clients qui n'ont encore rien payé. La stratégie naturelle est de remplir avec **0**.

In [ ]:
# Fill the NaN in TotalCharges with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Verify
assert df["TotalCharges"].isna().sum() == 0


## 4. Distribution de la cible

La variable cible est `Churn` (Yes/No). On veut savoir :
- Quel est le **taux de churn** global ?
- Le dataset est-il **équilibré** ?

Cela conditionne :
- Le choix des **métriques** (l'accuracy peut être trompeuse sur un dataset déséquilibré).
- La nécessité éventuelle de techniques de **rééquilibrage** (qu'on verra en S2).

In [ ]:
# Compute and display the value counts of `Churn`
print(df["Churn"].value_counts())

# Compute the churn rate (% of "Yes")
churn_rate = (df["Churn"] == "Yes").mean()
print(f"\nChurn rate: {churn_rate:.2%}")

# Plot a bar chart of the Churn distribution
plt.figure(figsize=(5, 4))
df["Churn"].value_counts().plot(kind="bar", color=["steelblue", "indianred"])
plt.title("Churn distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


## 5. Variables numériques vs target

Les 3 colonnes numériques sont `tenure`, `MonthlyCharges`, `TotalCharges`. On veut voir si leur distribution diffère selon que le client churn ou non — c'est le premier signal de **pouvoir prédictif**.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

# For each numeric column, plot two overlapping histograms (one per Churn class)
# Hint: use a 1x3 subplot, alpha=0.5, label each class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    for churn_value, label in [("No", "No churn"), ("Yes", "Churn")]:
        ax.hist(df.loc[df["Churn"] == churn_value, col], bins=30, alpha=0.5, label=label)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()


💡 **Lecture rapide** :
- `tenure` : les churners ont des tenures plus courtes (départs précoces).
- `MonthlyCharges` : les churners paient plus cher par mois — hypothèse : facturation perçue comme excessive.
- `TotalCharges` : suit `tenure × MonthlyCharges` — corrélation attendue, à garder en tête pour la multicolinéarité.

## 6. Variables catégorielles vs target

Pour chaque colonne catégorielle, on calcule le **taux de churn par modalité**. Une grosse variation entre modalités = signal prédictif fort.

In [ ]:
cat_cols = [c for c in df.columns
            if df[c].dtype == "object" and c not in ("customerID", "Churn")]
print(f"{len(cat_cols)} categorical columns: {cat_cols}")

# Write a helper function `churn_rate_by(col)` that returns
# the churn rate per category for a given column, sorted descending.
def churn_rate_by(col):
    return (
        df.groupby(col)["Churn"]
        .apply(lambda s: (s == "Yes").mean())
        .sort_values(ascending=False)
    )

# Apply it to "Contract", "InternetService", "PaymentMethod" and print the results.
# What do you observe? Which categories are the riskiest?
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"\nChurn rate by {col}:")
    print(churn_rate_by(col))


💡 **Insights métier majeurs** :
- **Contract** : les `Month-to-month` churnent ~3 fois plus que les `Two year`. Logique : pas d'engagement = facilité de départ.
- **InternetService** : `Fiber optic` churn beaucoup plus que `DSL`. Hypothèse : prix perçu trop élevé pour la qualité.
- **PaymentMethod** : `Electronic check` est sur-représenté dans les churners. Hypothèse : signal de friction (paiement manuel mensuel).

🧠 **Réflexe Tech Lead** : ces insights ne sont pas que pour le modèle. Ils alimentent directement les décisions métier (changer la stratégie d'engagement, revoir le pricing fibre, pousser le prélèvement automatique).

## 7. Synthèse de votre EDA

🎯 **Synthèse** :

> **Hypothèses retenues** :
> 1. `Contract = Month-to-month` et `PaymentMethod = Electronic check` sont des signaux forts de churn, donc ces variables devraient aider le modèle.
> 2. Les clients avec `tenure` faible et `MonthlyCharges` élevés sont plus à risque, donc `tenure` et `MonthlyCharges` doivent être des features importantes.
> 3. `InternetService = Fiber optic` est corrélé au churn, ce qui indique une dynamique de satisfaction/prix que le modèle peut capturer.
> 4. Le nombre de services (`services_count`) et l’engagement du client (`Contract`) sont des leviers pour distinguer les clients stables des churners.
> 5. Le mode de paiement et le comportement de facturation sont des proxys de friction client, utiles pour la prédiction.

## 8. Train / Validation / Test split

🚨 **Règle d'or** : on splitte **AVANT** tout traitement statistique sur la donnée (imputation, scaling, encoding). Sinon, on a une **fuite** depuis le test set vers le train set.

Le split classique :
- **Train (70 %)** : entraînement du modèle.
- **Validation (15 %)** : choix d'hyperparamètres, comparaison de modèles.
- **Test (15 %)** : score final, vu **une seule fois** à la fin du projet.

On utilise **`stratify=y`** pour conserver le taux de churn dans chaque split (sinon, on peut avoir un split de validation avec un taux de churn très différent, ce qui fausse l'évaluation).

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = (df["Churn"] == "Yes").astype(int)

# First split: train (70%) vs temp (30%), stratified, random_state=RANDOM_STATE
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

# Second split: temp -> val (50%) and test (50%), stratified
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):>5} rows | churn rate: {y_train.mean():.2%}")
print(f"Val:   {len(X_val):>5} rows | churn rate: {y_val.mean():.2%}")
print(f"Test:  {len(X_test):>5} rows | churn rate: {y_test.mean():.2%}")


## 9. Sauvegarde des splits

On persiste les splits sur disque pour pouvoir les recharger en TP2 sans tout refaire.

In [ ]:
Path("data").mkdir(exist_ok=True)
X_train.to_csv("data/X_train.csv", index=False)
X_val.to_csv("data/X_val.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_frame("Churn").to_csv("data/y_train.csv", index=False)
y_val.to_frame("Churn").to_csv("data/y_val.csv", index=False)
y_test.to_frame("Churn").to_csv("data/y_test.csv", index=False)
print("Splits saved to ./data/")


## 🚀 Bonus (pour les rapides)

1. **Heatmap de corrélation** : encodez `Churn` en 0/1 et calculez la matrice de corrélation entre les colonnes numériques + churn binaire. Visualisez avec `sns.heatmap`.
2. **Pairplot ciblé** : faites un `sns.pairplot(df[num_cols + ["Churn"]], hue="Churn")` et commentez.
3. **Multicolinéarité** : `TotalCharges ≈ tenure × MonthlyCharges`. Calculez la corrélation. À votre avis, faut-il garder les 3 features ? Justifiez (indice : ça dépend du modèle).
4. **Catégorielles cachées** : certaines colonnes "object" ont 3 modalités dont `"No internet service"` ou `"No phone service"`. Sont-elles vraiment 3 modalités ou plutôt des doublons d'un autre signal ? Inspectez.